# Laboratorium 3. Parametryzacja sygnału jako inżynieria cech: co „widzi” model?
**Głębokie Przetwarzanie Tekstu i Mowy | Politechnika Gdańska | sem. zimowy 2026/2027**

---

| | |
|---|---|
| **Imię i nazwisko (osoba A):** | **Igor Józefowicz** |
| **Nr indeksu:** | **193257** |
| **Imię i nazwisko (osoba B):** | *[wpisz tutaj]* |
| **Nr indeksu:** | *[wpisz tutaj]* |
| **Data wykonania:** | 04.11.2026 |
| **Deadline do wykonania:** | do 06.01.2027 |

---

### Cel laboratorium
Na LAB2 nagraliście we dwoje trzy korpusy mowy. Dziś będą one zbiorem danych do uczenia maszynowego. Sprawdzimy, jak
wybór reprezentacji (log-mel, MFCC, CQT), sposób podziału danych i normalizacja wpływają na to, czego model się
nauczy. Model nie słyszy nagrania. Dostaje tylko liczby, które z niego policzyliśmy, więc to, co odrzucimy na etapie
cech, jest dla niego stracone.

Zadajemy modelom dwa pytania: *co* zostało powiedziane (które zdanie) i *kto* to powiedział (osoba A czy B).
Przydadzą się do tego Wasze korpusy:

| Korpus | Kto co czytał | Do czego go użyjemy |
|---|---|---|
| 02 biometria | oboje te same 11 zdań (PL) | to samo zdanie u dwóch osób, wyciek danych |
| 03 ELRA | oboje te same zdania i krótkie frazy (EN) | rozpoznawanie zdań drugiej osoby, test w innym języku |
| 01 medyczne | każde inne 40 zdań (PL) | rozpoznawanie osoby niezależnie od tekstu |

### Pojęcia, które się przydadzą

| Pojęcie | Znaczenie |
|---|---|
| wektor cech | liczby opisujące jedno nagranie; klasyfikator widzi tylko je |
| zbiór uczący i testowy | na pierwszym model się uczy, na drugim sprawdzamy, jak radzi sobie z nowymi danymi |
| walidacja krzyżowa | dzielimy dane kilka razy na część uczącą i testową i uśredniamy wynik |
| wyciek danych | test jest za łatwy, bo w zbiorze uczącym były prawie te same przykłady co w testowym |
| niezmienniczość | cechy się nie zmieniają, gdy zmienia się coś nieistotnego dla zadania (np. wysokość głosu przy rozpoznawaniu zdania) |
| skrót (*shortcut learning*) | model rozwiązuje zadanie dzięki przypadkowej wskazówce (np. poziomowi nagrania) zamiast tej, o którą nam chodzi |
| przesunięcie rozkładu | dane we wdrożeniu różnią się od uczących (inny mikrofon, szum, kodek) |
| augmentacja | dokładamy do zbioru uczącego sztucznie zmienione kopie nagrań |

### Jak pracujemy
1. **Przewiduj.** Zanim uruchomisz komórkę z eksperymentem, zapisz w 1–2 zdaniach, czego się spodziewasz.
2. **Uzupełnij.** W każdym zadaniu jest jedna komórka z `TODO` (1–2 linie kodu). Miejsce na Twój kod jest
   oznaczone w komórce liniami `# ===== TODO n ... =====` i `# ===== koniec TODO n =====`; linię
   `raise NotImplementedError` zastąp swoim kodem. Pod kodem jest test: jeśli przechodzi, idź dalej.
3. **Wyjaśnij.** Odpowiedz na pytania, odwołując się do Waszych liczb i wykresów oraz do wykładu.

Komórka „Funkcje pomocnicze” zawiera gotowy kod (ekstrakcja cech, wykresy, walidacja). Nie musisz go czytać.

### Punktacja

| Zadanie | Temat | Punkty |
|---------|-------|--------|
| Zadanie 1 | Wasze nagrania jako zbiór danych | 1 |
| Zadanie 2 | To samo zdanie w trzech reprezentacjach | 1 |
| Zadanie 3 | Wektor cech i wyciek danych | 2 |
| Zadanie 4 | *Co* mówi: rozpoznawanie zdań drugiej osoby | 1 |
| Zadanie 5 | *Kto* mówi: rozpoznawanie osoby w innym języku | 1 |
| Zadanie 6 | Normalizacja, czyli co wyrzucić | 1 |
| Zadanie 7 | Zmiana warunków nagrania, skróty i augmentacja | 2 |
| Podsumowanie | Tabela wniosków | 1 |

### Zasady oceniania
- Kod z `TODO` musi przechodzić testy, bo bez tego kolejne kroki nie zadziałają.
- Oceniamy jakość rozumowania: czy wyjaśnienie odwołuje się do Waszych wyników, do wykładu i do pojęć z tabeli
  powyżej. Każda para ma inne nagrania, więc liczby będą się różnić między parami.
- Przewidywania muszą być wpisane, ale nie muszą być trafne. Punktujemy wyjaśnienie, dlaczego wynik okazał się inny
  niż hipoteza. Odpowiedzi „bo tak wyszło” nie dostają punktów.

## Konfiguracja środowiska
Obok notatnika muszą leżeć dwa foldery:

- **`nagrania/`**: Wasze nagrania z LAB2. Skopiuj folder `nagrania` z rejestratora albo połóż obok
  notatnika obie Wasze paczki `<indeks>.zip`, a notatnik rozpakuje je do `nagrania/`. Pliki `<indeks>_<fraza>_<korpus>.wav`
  i `metadata.csv` znajdzie sam, niezależnie od układu podfolderów.
- **`Dane/`**: kilka nagrań przykładowych (śpiewane gamy, nagranie po kompresji MP3).

Jeśli nagrywałeś sam albo brakuje nagrań drugiej osoby, poproś prowadzącego o nagrania innej osoby. Uruchom poniższe
komórki raz na początku zajęć. Jeśli pliki MP3 się nie wczytują, zaktualizuj `soundfile` (`pip install -U soundfile`).

In [1]:
%pip install -q librosa scikit-learn pandas seaborn matplotlib soundfile

Note: you may need to restart the kernel to use updated packages.


In [7]:
import re, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.fft
import scipy.signal
import librosa
from IPython.display import Audio, display

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, cross_val_predict
from sklearn.metrics import balanced_accuracy_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
RANDOM_STATE = 42
# VS Code udostepnia sciezke notebooka; w Jupyterze szukamy go w katalogu
# roboczym i jego katalogach nadrzednych (kernel zwykle startuje obok notebooka).
notebook_path = globals().get('__vsc_ipynb_file__')
if notebook_path:
    BASE_DIR = Path(notebook_path).resolve().parent
else:
    BASE_DIR = next(
        (folder for folder in (Path.cwd(), *Path.cwd().parents)
         if (folder / 'LAB3_parametryzacja_ML_student.ipynb').is_file()),
        None,
    )
    if BASE_DIR is None:
        raise FileNotFoundError(
            'Nie ustalono katalogu notebooka. Uruchom Jupyter z folderu '
            'zawierajacego ten notebook.'
        )

NAGRANIA, DANE = BASE_DIR / 'nagrania', BASE_DIR / 'Dane'
print(f"Katalog danych: {BASE_DIR}")

# paczki <indeks>.zip obok notatnika rozpakowujemy do nagrania/
for z in sorted(BASE_DIR.glob('*.zip')):
    if re.fullmatch(r'\d+', z.stem) and not (NAGRANIA / z.stem).exists():
        print(f"Rozpakowuję {z.name} do {NAGRANIA}/"); shutil.unpack_archive(z, NAGRANIA)

print(f"librosa {librosa.__version__}")
print(f"plików WAV w {NAGRANIA}: {len(list(NAGRANIA.rglob('*.wav')))}")

Katalog danych: C:\Users\igor\source\multimodal-ai\text-and-voice-processing\lab3
Rozpakowuję 193257.zip do C:\Users\igor\source\multimodal-ai\text-and-voice-processing\lab3\nagrania/
Rozpakowuję 193740.zip do C:\Users\igor\source\multimodal-ai\text-and-voice-processing\lab3\nagrania/
librosa 1.0.0
plików WAV w C:\Users\igor\source\multimodal-ai\text-and-voice-processing\lab3\nagrania: 432


### Funkcje pomocnicze
Uruchom. Nie musisz czytać tego kodu: w zadaniach wywołujemy te funkcje jednym poleceniem. Wszystkie nagrania
wczytujemy z częstotliwością próbkowania 16 kHz i obcinamy ciszę na początku i końcu. Ramka ma 25 ms, kolejne ramki
są co 10 ms (100 ramek na sekundę).

In [8]:
SR = 16000
N_FFT, WIN, HOP, N_MELS, N_MFCC = 512, 400, 160, 40, 13      # FFT 32 ms, okno 25 ms, krok 10 ms
CQT_FMIN, CQT_BINS = librosa.note_to_hz('C2'), 72            # 6 oktaw: C2 (65 Hz) … H7 (ok. 4 kHz)
KINDS = ['log-mel', 'MFCC', 'CQT']
RAMEK_NA_S = SR // HOP


def wczytaj(sciezka, sekundy=None):
    # nagranie mono 16 kHz bez ciszy na początku i końcu
    y = librosa.load(sciezka, sr=SR, duration=sekundy)[0]
    return librosa.effects.trim(y, top_db=35)[0]


def extract(y, kind):
    # macierz cech jednego nagrania: wiersze to ramki, kolumny to cechy
    if kind == 'CQT':
        return librosa.amplitude_to_db(np.abs(librosa.cqt(y, sr=SR, hop_length=HOP, fmin=CQT_FMIN, n_bins=CQT_BINS))).T
    L = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, win_length=WIN,
                                                           hop_length=HOP, n_mels=N_MELS)).T
    return L if kind == 'log-mel' else mfcc_from_logmel(L)      # mfcc_from_logmel: Twoja funkcja z Zadania 2


def pokaz(nagrania):
    # nagrania: słownik {tytuł: sygnał}; rysuje log-mel, MFCC (bez c0) i CQT, kolumny to nagrania
    fig, ax = plt.subplots(len(KINDS), len(nagrania), figsize=(4 * len(nagrania), 8), squeeze=False)
    for j, (tytul, y) in enumerate(nagrania.items()):
        for r, k in enumerate(KINDS):
            F = extract(y, k)[:, 1:] if k == 'MFCC' else extract(y, k)   # c0 (głośność) zdominowałby kolory
            ax[r, j].imshow(F.T, aspect='auto', origin='lower', cmap='magma'); ax[r, j].grid(False)
            ax[r, j].set_title(f"{k}{' c1-c12' if k == 'MFCC' else ''}: {tytul}", fontsize=8)
            ax[r, j].set(xlabel='ramka (co 10 ms)' if r == len(KINDS) - 1 else '', ylabel='cecha' if j == 0 else '')
    plt.tight_layout(); plt.show()


def make_clf():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))


def evaluate(X, y, groups=None):
    # walidacja krzyżowa; zwraca (zrównoważona dokładność, predykcje)
    # groups=None: losowy podział na 5 części; groups=tablica grup: każda grupa raz jest całym zbiorem testowym
    cv = LeaveOneGroupOut() if groups is not None else StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
    pred = cross_val_predict(make_clf(), X, y, groups=groups, cv=cv)
    return balanced_accuracy_score(y, pred), pred


def train_test(X, y, tr, te):
    # uczy na wierszach tr, testuje na wierszach te (maski logiczne); zwraca zrównoważoną dokładność
    return balanced_accuracy_score(y[te], make_clf().fit(X[tr], y[tr]).predict(X[te]))


def fragmenty(F, L, krok=None):
    # dzieli macierz cech [ramki × cechy] na fragmenty po L ramek przesuwane co `krok` ramek (domyślnie bez zakładki)
    return [F[i:i + L] for i in range(0, len(F) - L + 1, krok or L)]


def wspolne_nr(korpus, od=1):
    # numery fraz z danego korpusu, które nagrały obie osoby
    nr = [set(rec.nr[(rec.osoba == o) & (rec.korpus == korpus) & (rec.nr >= od)]) for o in 'AB']
    return sorted(nr[0] & nr[1])


def indeks(osoba, korpus, nr):
    # numer wiersza w tabeli rec dla danej osoby, korpusu i frazy
    return rec.index[(rec.osoba == osoba) & (rec.korpus == korpus) & (rec.nr == nr)][0]


NOISE_RNG = np.random.default_rng(RANDOM_STATE)


def add_noise(y, snr_db):
    # dodaje szum biały o zadanym stosunku sygnału do szumu [dB]
    n = NOISE_RNG.standard_normal(len(y))
    return y + n * np.sqrt(np.mean(y ** 2) / (np.mean(n ** 2) * 10 ** (snr_db / 10)))

### Wczytanie Waszych nagrań
Komórka zbiera wszystkie nagrania i metadane z rejestratora w jedną tabelę `rec` (jeden wiersz to jedno nagranie) i
wczytuje sygnały do listy `signals` (w tej samej kolejności). Nagrania oznaczone przez rejestrator jako błędne
pomijamy. Jeśli w folderze są nagrania innych osób, wpisz Wasze numery indeksów w `MOWCY`.

In [ ]:
MOWCY = None          # np. ['213769', '148869']; None = dwie osoby z największą liczbą nagrań

pliki = sorted(p for p in NAGRANIA.rglob('*.wav') if '_odrzucone' not in p.parts and re.fullmatch(r'\d+_\d+_\d\d', p.stem))
assert pliki, f"Brak nagrań w {NAGRANIA.resolve()}: skopiuj tam folder nagrania albo połóż obok notatnika paczki <indeks>.zip."
meta = pd.concat([pd.read_csv(m, sep=';', dtype=str, encoding='utf-8-sig') for m in NAGRANIA.rglob('metadata.csv')
                  if '_odrzucone' not in m.parts], ignore_index=True).drop_duplicates('plik')
rec = pd.DataFrame({'sciezka': pliki, 'plik': [p.name for p in pliki]}).drop_duplicates('plik')
rec[['mowca', 'nr', 'korpus']] = rec['plik'].str.extract(r'^(\d+)_(\d+)_(\d\d)\.wav$')
rec = rec.merge(meta[['plik', 'tekst', 'status_qc', 'dlugosc_s', 'poziom_mowy_dbfs', 'szum_dbfs', 'snr_db']],
                on='plik', how='left')
for c in ['dlugosc_s', 'poziom_mowy_dbfs', 'szum_dbfs', 'snr_db']:
    rec[c] = pd.to_numeric(rec[c], errors='coerce')
print(f"Pominięte nagrania z błędem QC: {(rec['status_qc'] == 'blad').sum()}")
rec = rec[rec['status_qc'] != 'blad']

MOWCY = MOWCY or rec['mowca'].value_counts().index[:2].tolist()
assert len(MOWCY) == 2, "Potrzebne są nagrania dwóch osób. Poproś prowadzącego o nagrania drugiej osoby."
rec = rec[rec['mowca'].isin(MOWCY)].reset_index(drop=True)
rec['osoba'] = rec['mowca'].map({MOWCY[0]: 'A', MOWCY[1]: 'B'})
rec['nr'] = rec['nr'].astype(int)
rec['typ'] = np.select([rec['korpus'] == '01', rec['korpus'] == '02', (rec['korpus'] == '03') & (rec['nr'] >= 78)],
                       ['medyczne (PL)', 'biometria (PL)', 'ELRA zdania (EN)'], 'ELRA krótkie (EN)')
rec['jezyk'] = np.where(rec['korpus'] == '03', 'EN', 'PL')

print(f"Osoba A = {MOWCY[0]}, osoba B = {MOWCY[1]}. Wczytuję {len(rec)} nagrań (ok. 1 min)…")
signals = [wczytaj(p) for p in rec['sciezka']]
display(pd.crosstab(rec['typ'], rec['osoba'], margins=True, margins_name='razem'))
assert min(rec['osoba'].value_counts()) >= 50, "Jedna z osób ma mniej niż 50 nagrań: sprawdź MOWCY i folder nagrania/."

---
## Zadanie 1: Wasze nagrania jako zbiór danych (1 pkt)

Zanim zaczniemy uczyć modele, trzeba wiedzieć, co jest w danych. Rejestrator zapisał dla każdego nagrania długość,
poziom mowy, poziom szumu i SNR.

### Krok 1: Podsumowanie zbioru
**TODO 1.** Uzupełnij tabelę `podsumowanie` o dwie kolumny: łączny czas nagrań w minutach i medianę poziomu mowy. Wzór masz w
linii z kolumną `nagran`: `nazwa=('kolumna', funkcja)`, gdzie funkcja to napis (np. `'median'`) albo `lambda`.

In [ ]:
podsumowanie = rec.groupby(['osoba', 'typ']).agg(
    nagran=('plik', 'size'),
    # ===== TODO 1 (2 linie): wpisz kod poniżej =====
    # czas_min: łączny czas nagrań w minutach (kolumna 'dlugosc_s'); poziom_mowy: mediana 'poziom_mowy_dbfs'

    # ===== koniec TODO 1 =====
)

# test
assert {'czas_min', 'poziom_mowy'} <= set(podsumowanie.columns), "dodaj kolumny czas_min i poziom_mowy"
assert np.isclose(podsumowanie['czas_min'].sum(), rec['dlugosc_s'].sum() / 60), "czas_min: suma kolumny dlugosc_s w minutach"
print("TODO 1 OK"); display(podsumowanie.round(1))

### Krok 2: Czym różnią się Wasze nagrania poza głosem?
Komórka rysuje rozkłady poziomu mowy, szumu i SNR dla każdej osoby i szacuje wysokość głosu F0. Potem uczy
klasyfikator, który rozpoznaje osobę **wyłącznie z metadanych** (poziom mowy, poziom szumu, SNR), bez dostępu do
dźwięku. Poziom losowy to 0,5.

> **Przewidywanie 1** *(zapisz przed uruchomieniem kolejnej komórki; za błędną hipotezę nie traci się punktów)*: Czy da się rozpoznać, kto z Was mówi, na podstawie samych metadanych nagrania? Jaki wynik przewidujesz?
>
> *Moja hipoteza:*

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
for a, c in zip(ax, ['poziom_mowy_dbfs', 'szum_dbfs', 'snr_db']):
    sns.boxplot(data=rec, x='osoba', y=c, hue='osoba', ax=a, legend=False)
plt.tight_layout(); plt.show()

f0 = {o: np.nanmedian(np.concatenate([librosa.pyin(signals[i], fmin=60, fmax=500, sr=SR)[0]
                                      for i in rec[rec.osoba == o].sample(8, random_state=0).index])) for o in 'AB'}
display(rec.groupby('osoba')[['poziom_mowy_dbfs', 'szum_dbfs', 'snr_db']].median().assign(**{'F0 [Hz]': pd.Series(f0)}).round(1))

ok = rec[['poziom_mowy_dbfs', 'szum_dbfs', 'snr_db']].notna().all(axis=1).to_numpy()
acc_meta, _ = evaluate(rec.loc[ok, ['poziom_mowy_dbfs', 'szum_dbfs', 'snr_db']].to_numpy(), rec.loc[ok, 'osoba'].to_numpy())
print(f"Rozpoznawanie osoby z samych metadanych: {acc_meta:.3f} (poziom losowy 0,5)")

### Pytania do Zadania 1

**P1.1** Opisz Wasz zbiór: ile nagrań i minut ma każda osoba w każdym typie, czy zbiór jest zrównoważony, ile nagrań odpadło przez QC. Które nagrania czytaliście oboje, a które są różne dla każdej osoby? Dlaczego zdania medyczne nie nadają się do sprawdzania, *co* zostało powiedziane, choć nadają się do rozpoznawania, *kto* mówi?

**P1.2** Czym różnią się Wasze nagrania poza samym głosem? Ile wyniósł klasyfikator na samych metadanych i co to oznacza dla klasyfikatora, który dostanie cechy MFCC? Która z trzech wielkości (poziom mowy, szum, SNR) różni Was najbardziej? Skąd mogą się brać takie różnice, nawet jeśli nagrywaliście na tym samym stanowisku? (Wrócimy do tego w Zadaniu 7.)

> **Twoja odpowiedź:**
>
> P1.1:
>
>
> P1.2:

---
## Zadanie 2: To samo zdanie w trzech reprezentacjach (1 pkt)

| Reprezentacja | Liczb na ramkę | Co zachowuje | Co odrzuca |
|---|---|---|---|
| log-mel | 40 | energię w pasmach melowych (gęsto w niskich częstotliwościach, rzadko w wysokich) | szczegóły widma w wysokich częstotliwościach |
| MFCC | 13 | DCT z log-mel, czyli obwiednię widma (formanty, „barwę” głosu) | drobną strukturę widma: harmoniczne, czyli wysokość głosu F0 |
| CQT | 72 | widmo w półtonach od C2 do H7 (od 65 Hz do ok. 4 kHz); rozdziela harmoniczne | wszystko powyżej ok. 4 kHz |

### Krok 1: MFCC z log-mel
Z wykładu: MFCC to DCT z logarytmów energii pasm melowych. **TODO 2.** Uzupełnij funkcję `mfcc_from_logmel` jedną linią z
`scipy.fft.dct`. Wiersze macierzy to ramki, a kolumny to pasma, więc DCT liczymy wzdłuż kolumn (`axis=1`), z
`type=2` i `norm='ortho'`. Potem zostawiamy `n_mfcc` pierwszych kolumn.

In [ ]:
def mfcc_from_logmel(L, n_mfcc=N_MFCC):
    # L: log-mel [ramki × pasma] w dB  ->  MFCC [ramki × n_mfcc]
    # ===== TODO 2 (1 linia): wpisz kod w miejsce linii raise =====
    # DCT z L wzdłuż kolumn (parametry w opisie nad komórką), zwróć n_mfcc pierwszych kolumn
    raise NotImplementedError("TODO 2: uzupełnij kod")
    # ===== koniec TODO 2 =====


# test: porównanie z librosa.feature.mfcc na Waszym pierwszym nagraniu
ref = librosa.feature.mfcc(y=signals[0], sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT, win_length=WIN, hop_length=HOP, n_mels=N_MELS).T
assert extract(signals[0], 'MFCC').shape == ref.shape, "zły kształt: wiersze to ramki, kolumny to współczynniki"
assert np.allclose(extract(signals[0], 'MFCC'), ref, atol=1e-3), "wartości różnią się od librosa: sprawdź oś i normalizację DCT"
print("TODO 2 OK: Twoje MFCC są takie same jak w librosa")

### Krok 2: Wy dwoje i dwie śpiewane gamy
Cztery nagrania (pierwsze 6 s): to samo zdanie z korpusu biometrii w wykonaniu A i B oraz dwie gamy z folderu
`Dane/`: mężczyzna śpiewa na samogłosce „e”, kobieta na „a”. W gamie samogłoska jest stała, a wysokość dźwięku rośnie.
Najpierw posłuchaj, potem obejrzyj wykresy.

> **Przewidywanie 2** *(zapisz przed uruchomieniem kolejnej komórki; za błędną hipotezę nie traci się punktów)*: W której reprezentacji melodię gamy widać najwyraźniej, a w której prawie jej nie widać? W której reprezentacji Wasze dwa nagrania tego samego zdania będą najbardziej podobne?
>
> *Moja hipoteza:*

In [ ]:
NR_ZDANIA = wspolne_nr('02')[0]           # możesz wybrać inne zdanie biometrii (1–11)
cztery = {f'A: biometria {NR_ZDANIA}': signals[indeks('A', '02', NR_ZDANIA)][:6 * SR],
          f'B: biometria {NR_ZDANIA}': signals[indeks('B', '02', NR_ZDANIA)][:6 * SR],
          'gama męska (Dane)': wczytaj(DANE / 'm2_scales_straight_e.wav', 6),
          'gama żeńska (Dane)': wczytaj(DANE / 'f7_scales_straight_a.wav', 6)}
print(rec.loc[indeks('A', '02', NR_ZDANIA), 'tekst'])
for tytul, y in cztery.items():
    print(tytul); display(Audio(y, rate=SR))
pokaz(cztery)

### Pytania do Zadania 2

**P2.1** Porównaj przewidywanie z wykresami. Wasze nagrania tego samego zdania: w której reprezentacji są najbardziej podobne, a w której najbardziej się różnią? Jak wygląda męska gama w CQT, a jak w MFCC? Której reprezentacji użyłbyś, gdyby model miał rozpoznawać, *co* zostało powiedziane, a której, gdyby miał rozpoznawać melodię albo wysokość głosu?

**P2.2** Gama żeńska: czy MFCC są nadal prawie stałe, tak jak przy gamie męskiej? Spróbuj wyjaśnić różnicę. Wskazówka: obejrzyj na wykresie log-mel, jak wyglądają harmoniczne gamy żeńskiej w najniższych pasmach. Co to może oznaczać dla modelu mowy, który uczono głównie na niższych głosach?

> **Twoja odpowiedź:**
>
> P2.1:
>
>
> P2.2:

---
## Zadanie 3: Wektor cech i wyciek danych (2 pkt)

Klasyfikator potrzebuje wektora tej samej długości dla każdego przykładu, a nagrania mają różną liczbę ramek.
Najprostsze rozwiązanie: dla każdej cechy policzyć średnią i odchylenie standardowe po wszystkich ramkach.

$$\text{nagranie}\;\rightarrow\;\underbrace{\text{macierz cech}}_{\text{ramki} \times F}\;\rightarrow\;\underbrace{[\text{średnie},\ \text{odchylenia}]}_{2F \text{ liczb}}\;\rightarrow\;\text{klasyfikator}$$

### Krok 1: Wektor cech
**TODO 3.** Uzupełnij `pool_meanstd` jedną linią: połącz (`np.concatenate`) średnie kolumn i odchylenia standardowe kolumn
(`mean` i `std` wzdłuż osi 0). Komórka policzy też macierze cech wszystkich nagrań w trzech reprezentacjach
(słownik `FR`, ok. 1–3 min); przydadzą się w kolejnych zadaniach.

In [ ]:
def pool_meanstd(F):
    # F [ramki × cechy]  ->  wektor [średnie kolumn..., odchylenia std kolumn...] o długości 2·F.shape[1]
    # ===== TODO 3 (1 linia): wpisz kod w miejsce linii raise =====
    # połącz w jeden wektor średnie kolumn F i odchylenia std kolumn F
    raise NotImplementedError("TODO 3: uzupełnij kod")
    # ===== koniec TODO 3 =====


# test
assert np.allclose(pool_meanstd(np.array([[1., 10.], [3., 30.]])), [2, 20, 1, 10]), "zła kolejność lub oś"
print("TODO 3 OK; liczę cechy wszystkich nagrań…")
FR = {k: [extract(s, k) for s in signals] for k in KINDS}
print("gotowe")

### Krok 2: Które to zdanie? Fragmenty i dwa sposoby podziału
Tniemy nagrania biometrii (11 zdań, oboje czytaliście te same) na fragmenty po 1 s, przesuwane co 0,1 s. Fragmenty
zachodzą więc na siebie; tak często robi się z długimi sygnałami, żeby mieć więcej przykładów.
Klasyfikator ma rozpoznać, z którego zdania pochodzi fragment. Poziom losowy to 1/11 ≈ 0,09. Sprawdzamy go na dwa
sposoby:

- **losowy podział fragmentów**: fragmenty tasujemy i dzielimy na 5 części, więc w zbiorze uczącym są inne fragmenty
  *tego samego nagrania*,
- **test na drugiej osobie**: uczymy na fragmentach osoby A, testujemy na fragmentach osoby B (i odwrotnie).

> **Przewidywanie 3** *(zapisz przed uruchomieniem kolejnej komórki; za błędną hipotezę nie traci się punktów)*: Który sposób podziału da wyższy wynik i mniej więcej o ile?
>
> *Moja hipoteza:*

In [ ]:
L_FRAG, KROK_FRAG = 100, 10                          # długość i przesunięcie fragmentu w ramkach (100 ramek = 1 s)
bio = [indeks(o, '02', n) for n in wspolne_nr('02') for o in 'AB']
wyciek = {}
for k in KINDS:
    Xf, zdanie, osoba = [], [], []
    for i in bio:
        for frag in fragmenty(FR[k][i], L_FRAG, KROK_FRAG):
            Xf.append(pool_meanstd(frag)); zdanie.append(rec.nr[i]); osoba.append(rec.osoba[i])
    Xf, zdanie, osoba = np.array(Xf), np.array(zdanie), np.array(osoba)
    wyciek[k] = {'losowy podział fragmentów': evaluate(Xf, zdanie)[0],
                 'test na drugiej osobie': evaluate(Xf, zdanie, groups=osoba)[0]}
print(f"{len(Xf)} fragmentów z {len(bio)} nagrań")
wyciek = pd.DataFrame(wyciek).T
display(wyciek.round(3))
ax = wyciek.plot(kind='bar', rot=0, figsize=(7, 3.5), edgecolor='k', ylim=(0, 1))
ax.axhline(1 / 11, ls='--', c='gray', lw=1); ax.set(ylabel='zrównoważona dokładność', title='Które to zdanie?')
plt.show()

### Pytania do Zadania 3

**P3.1** Podaj wyniki dla trzech reprezentacji i porównaj oba sposoby podziału. Skąd bierze się różnica? Ile wspólnych ramek mają dwa sąsiednie fragmenty i co jeszcze łączy fragmenty jednego nagrania? Który wynik mówi coś o tym, jak model rozpozna to zdanie u nowej osoby? Ustaw `KROK_FRAG = 100` (bez zakładki), uruchom komórkę ponownie i porównaj wyniki. Czy wynik losowego podziału jest podobny we wszystkich reprezentacjach? Jeśli nie, zaproponuj wyjaśnienie. Na podstawie czego stwierdzasz, czy w tym eksperymencie wystąpił wyciek danych?

**P3.2** Podaj przykład z innej dziedziny, niż mowa, w którym dane też tnie się na fragmenty i losowy podział dałby za wysoki wynik. Według czego należałoby tam podzielić dane?

> **Twoja odpowiedź:**
>
> P3.1:
>
>
> P3.2:

---
## Zadanie 4: *Co* mówi: rozpoznawanie zdań drugiej osoby (1 pkt)

Budujemy najprostszy „rozpoznawacz mowy”. Nagrania zdań ELRA osoby A są wzorcami. Dla każdego nagrania osoby B
szukamy najbardziej podobnego wzorca i sprawdzamy, czy to to samo zdanie (i odwrotnie: B to wzorce, A to zapytania).
To klasyfikator najbliższego sąsiada. Nagrania mają różną długość i tempo, więc nie uśredniamy ramek, tylko
porównujemy całe sekwencje algorytmem DTW (*dynamic time warping*), który dopasowuje ramki w czasie. Tak działały
pierwsze systemy rozpoznawania słów.

### Krok 1: Odległość DTW
**TODO 4.** Uzupełnij `dtw_distance` dwiema liniami: wywołaj `librosa.sequence.dtw(X=..., Y=..., metric='euclidean')`, która
zwraca macierz skumulowanych kosztów `D` i ścieżkę dopasowania. librosa oczekuje cech w wierszach, więc podaj
macierze transponowane (`Fa.T`, `Fb.T`). Zwróć koszt całej ścieżki `D[-1, -1]` podzielony przez jej długość.

In [ ]:
def dtw_distance(Fa, Fb):
    # Fa, Fb: macierze cech [ramki × cechy] dwóch nagrań  ->  średni koszt na krok ścieżki dopasowania
    # ===== TODO 4 (2 linie): wpisz kod w miejsce linii raise =====
    # D i ścieżka z librosa.sequence.dtw na macierzach transponowanych; zwróć D[-1, -1] / długość ścieżki
    raise NotImplementedError("TODO 4: uzupełnij kod")
    # ===== koniec TODO 4 =====


# test: nagranie porównane samo ze sobą ma koszt 0, a z innym nagraniem większy
assert np.isclose(dtw_distance(FR['MFCC'][0], FR['MFCC'][0]), 0), "nagranie porównane samo ze sobą powinno mieć koszt 0"
assert dtw_distance(FR['MFCC'][0], FR['MFCC'][1]) > 0
print("TODO 4 OK")

### Krok 2: Rozpoznawanie zdań ELRA drugiej osoby
Porównujemy cztery reprezentacje; „MFCC bez c0” to MFCC bez pierwszego współczynnika, który odpowiada głównie za
głośność. Mapa pokazuje koszty DTW (jaśniej = bardziej podobne po normalizacji w wierszu): poprawne dopasowania leżą
na przekątnej.

> **Przewidywanie 4** *(zapisz przed uruchomieniem kolejnej komórki; za błędną hipotezę nie traci się punktów)*: Która reprezentacja najlepiej rozpozna zdania drugiej osoby, a która najgorzej? Czy usunięcie $c_0$ pomoże?
>
> *Moja hipoteza:*

In [ ]:
N_ZDAN = 40                                          # liczba zdań ELRA (maks. 88); więcej = dokładniej, ale dłużej
nr_zdan = wspolne_nr('03', od=78)[:N_ZDAN]
idx = {o: [indeks(o, '03', n) for n in nr_zdan] for o in 'AB'}
REPS = {'log-mel': ('log-mel', slice(None)), 'MFCC': ('MFCC', slice(None)),
        'MFCC bez c0': ('MFCC', slice(1, None)), 'CQT': ('CQT', slice(None))}


def dopasowanie(frames, cols):
    # macierz kosztów DTW: wiersze to nagrania B, kolumny to nagrania A
    return np.array([[dtw_distance(frames[b][:, cols], frames[a][:, cols]) for a in idx['A']] for b in idx['B']])


def trafnosc(D):
    # odsetek poprawnie rozpoznanych zdań, średnio w obu kierunkach (B wśród wzorców A i A wśród wzorców B)
    ok = np.arange(len(D))
    return (np.mean(D.argmin(axis=1) == ok) + np.mean(D.argmin(axis=0) == ok)) / 2


DTW = {r: dopasowanie(FR[k], cols) for r, (k, cols) in REPS.items()}
print(f"{len(nr_zdan)} zdań, poziom losowy {1 / len(nr_zdan):.3f}")
display(pd.Series({r: trafnosc(D) for r, D in DTW.items()}, name='trafność').round(3).to_frame())

D = DTW['MFCC bez c0']
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(-(D - D.mean(axis=1, keepdims=True)) / D.std(axis=1, keepdims=True), cmap='magma', square=True,
            xticklabels=False, yticklabels=False, cbar=False)
plt.title('MFCC bez c0: koszty DTW (wiersze: B, kolumny: A)'); plt.show()

### Pytania do Zadania 4

**P4.1** Podaj wyniki. Która reprezentacja wypadła najlepiej, a która najgorzej? Wyjaśnij to, odwołując się do tego, co każda z nich zachowuje (Zadanie 2), i do tego, czym różnią się Wasze głosy (Zadanie 1). Jak zmienił wynik brak $c_0$ i co to mówi o różnicy poziomów Waszych nagrań? Które zdania były mylone (punkty jasne poza przekątną) i dlaczego akurat te?

**P4.2** W Zadaniu 3 uśrednialiśmy ramki, a tu porównujemy całe sekwencje. Jakie założenie o danych wprowadza DTW (obciążenie indukcyjne)? Dlaczego ten sposób rozpoznawania mowy nie skaluje się do prawdziwych systemów i co robią zamiast niego współczesne modele (wykład)?

> **Twoja odpowiedź:**
>
> P4.1:
>
>
> P4.2:

---
## Zadanie 5: *Kto* mówi: rozpoznawanie osoby w innym języku (1 pkt)

Teraz zadanie odwrotne: rozpoznajemy osobę (A czy B) z wektora `pool_meanstd` całego nagrania. Żeby test był
trudniejszy niż losowy podział, uczymy na nagraniach polskich (medyczne i biometria), a testujemy na angielskich
(ELRA). Model nie widział więc w treningu ani tych zdań, ani tego języka.

### Krok 1: Podział na trening i test
**TODO 5.** Uzupełnij jedną linię: `tr` to maska nagrań polskich, `te` maska nagrań angielskich (kolumna `rec['jezyk']`), obie jako
tablice numpy (`.to_numpy()`).

In [ ]:
X = {k: np.array([pool_meanstd(F) for F in FR[k]]) for k in KINDS}      # wektor cech każdego nagrania
y_osoba = rec['osoba'].to_numpy()
# ===== TODO 5 (1 linia): wpisz kod w miejsce linii raise =====
# tr = maska nagrań polskich, te = maska nagrań angielskich (kolumna rec['jezyk'], obie .to_numpy())
raise NotImplementedError("TODO 5: uzupełnij kod")
# ===== koniec TODO 5 =====

# test
assert tr.sum() > 0 and te.sum() > 0 and not (tr & te).any(), "zbiór uczący i testowy muszą być niepuste i rozłączne"
assert set(rec['jezyk'][tr]) == {'PL'} and set(rec['jezyk'][te]) == {'EN'}, "uczymy na polskich, testujemy na angielskich"
print(f"TODO 5 OK: {tr.sum()} nagrań uczących, {te.sum()} testowych")

### Krok 2: Całe nagrania i krótkie fragmenty
Pierwsza tabela porównuje losowy podział nagrań z testem w innym języku. Wykres pokazuje, jak wynik zależy od
długości fragmentu mowy, z którego rozpoznajemy osobę (uczenie i test na fragmentach tej samej długości).

> **Przewidywanie 5** *(zapisz przed uruchomieniem kolejnej komórki; za błędną hipotezę nie traci się punktów)*: Czy model nauczony na polskich nagraniach rozpozna Was w nagraniach angielskich? Ile sekund mowy wystarczy?
>
> *Moja hipoteza:*

In [ ]:
kto = pd.DataFrame({k: {'losowy podział nagrań': evaluate(X[k], y_osoba)[0],
                        'uczenie PL -> test EN': train_test(X[k], y_osoba, tr, te),
                        'uczenie EN -> test PL': train_test(X[k], y_osoba, te, tr)} for k in KINDS}).T
display(kto.round(3))

DLUGOSCI = [10, 30, 100, 300]                        # ramki: 0,1 s, 0,3 s, 1 s, 3 s
rows = []
for k in KINDS:
    for L in DLUGOSCI:
        Xf, yf, jf = [], [], []
        for F, o, j in zip(FR[k], y_osoba, rec['jezyk']):
            for frag in fragmenty(F, L):
                Xf.append(pool_meanstd(frag)); yf.append(o); jf.append(j)
        Xf, yf, jf = np.array(Xf), np.array(yf), np.array(jf)
        rows.append({'reprezentacja': k, 'fragment [s]': L / RAMEK_NA_S, 'trafność': train_test(Xf, yf, jf == 'PL', jf == 'EN')})
dl = pd.DataFrame(rows).pivot(index='fragment [s]', columns='reprezentacja', values='trafność')
display(dl.round(3))
ax = dl.plot(marker='o', logx=True, figsize=(6, 3.5), ylim=(0.4, 1.02))
ax.axhline(0.5, ls='--', c='gray', lw=1); ax.set(ylabel='zrównoważona dokładność', title='Uczenie PL -> test EN')
plt.show()

### Pytania do Zadania 5

**P5.1** Podaj wyniki dla całych nagrań. Czy rozpoznawanie osoby działa w innym języku niż ten, na którym model się uczył? Dlaczego? Czy wynik bliski 1,0 przy dwóch osobach znaczy, że mamy dobry system biometryczny dla całej firmy? Czego ten test nie sprawdza?

**P5.2** Jak wynik zależy od długości fragmentu i dlaczego? Która reprezentacja potrzebuje najmniej mowy? Jakie to ma znaczenie przy projektowaniu weryfikacji głosowej (zdania biometrii trwały ok. 7 s)?

> **Twoja odpowiedź:**
>
> P5.1:
>
>
> P5.2:

---
## Zadanie 6: Normalizacja, czyli co wyrzucić (1 pkt)

Normalizacja cech też decyduje o tym, jaką informację model dostanie. Porównamy dwa warianty z wykładu:

- **CMN per nagranie**: od każdej cechy odejmujemy jej średnią po wszystkich ramkach danego nagrania.
- **CMVN per osoba**: średnią $\mu$ i odchylenie $\sigma$ każdej cechy liczymy ze wszystkich ramek danej osoby i
  zamieniamy cechy na $(F - \mu)/\sigma$.

### Krok 1: Normalizacja per osoba
**TODO 6.** Uzupełnij dwie linie w `normalize_per_speaker`.

In [ ]:
def normalize_per_recording(frames):
    return [F - F.mean(axis=0) for F in frames]


def normalize_per_speaker(frames, speakers):
    out = [None] * len(frames)
    for s in np.unique(speakers):
        idx_s = np.flatnonzero(speakers == s)            # numery nagrań osoby s
        A = np.vstack([frames[i] for i in idx_s])        # wszystkie ramki osoby s [ramki × cechy]
        # ===== TODO 6 (2 linie): wpisz kod w miejsce linii raise =====
        # mu = średnia kolumn A, sd = odchylenie std kolumn A + 1e-8
        raise NotImplementedError("TODO 6: uzupełnij kod")
        # ===== koniec TODO 6 =====
        for i in idx_s:
            out[i] = (frames[i] - mu) / sd
    return out


# test
A_ = np.vstack([F for F, o in zip(normalize_per_speaker(FR['MFCC'], y_osoba), y_osoba) if o == 'A'])
assert np.allclose(A_.mean(axis=0), 0, atol=1e-4) and np.allclose(A_.std(axis=0), 1, atol=1e-3)
print("TODO 6 OK")

### Krok 2: Normalizacja a oba pytania
Pierwsza tabela: rozpoznawanie zdań drugiej osoby (DTW z Zadania 4). Druga: rozpoznawanie osoby (uczenie PL,
test EN, jak w Zadaniu 5). Obliczenia trwają ok. 1–2 min.

> **Przewidywanie 6** *(zapisz przed uruchomieniem kolejnej komórki; za błędną hipotezę nie traci się punktów)*: Jak każda normalizacja zmieni rozpoznawanie zdań, a jak rozpoznawanie osoby?
>
> *Moja hipoteza:*

In [ ]:
NORMY = {'brak': lambda fr: fr, 'CMN per nagranie': normalize_per_recording,
         'CMVN per osoba': lambda fr: normalize_per_speaker(fr, y_osoba)}

co = pd.DataFrame({r: {n: trafnosc(dopasowanie(norm(FR[REPS[r][0]]), REPS[r][1])) for n, norm in NORMY.items()}
                   for r in ['log-mel', 'MFCC bez c0']}).T
print(f"CO mówi: rozpoznawanie zdań drugiej osoby (DTW), poziom losowy {1 / len(nr_zdan):.3f}")
display(co.round(3))

kto_n = pd.DataFrame({k: {n: train_test(np.array([pool_meanstd(F) for F in norm(FR[k])]), y_osoba, tr, te)
                          for n, norm in NORMY.items()} for k in ['MFCC', 'CQT']}).T
print("KTO mówi: uczenie PL -> test EN, poziom losowy 0,5")
display(kto_n.round(3))

### Pytania do Zadania 6

**P6.1** Jak normalizacja zmieniła rozpoznawanie zdań? Co dokładnie usuwa każda z nich i jak to wpływa na dopasowanie tego samego zdania u dwóch osób? Sformułuj jednym zdaniem ogólną zasadę: o czym decydujemy, wybierając normalizację cech?

**P6.2** Co stało się z rozpoznawaniem osoby po CMVN per osoba i dlaczego? Czy takiej normalizacji można użyć w systemie rozpoznawania osoby, a czy w systemie rozpoznawania mowy dla znanego użytkownika? Uzasadnij: co trzeba wiedzieć o nagraniu testowym, żeby ją policzyć?

> **Twoja odpowiedź:**
>
> P6.1:
>
>
> P6.2:

---
## Zadanie 7: Zmiana warunków nagrania, skróty i augmentacja (2 pkt)

Model zwykle uczy się na jednych nagraniach, a działa na innych: przez telefon, w hałasie, z innym ustawieniem
wzmocnienia. To przesunięcie rozkładu. Sprawdzimy je w trzech krokach.

### Krok 1: Kodek MP3 (pliki z `Dane/`)
`Speaker_8_153.flac` to oryginał, a pliki `Speaker_8_153_8.mp3` … `Speaker_8_153_320.mp3` to to samo nagranie po
kompresji MP3 z przepływnością 8, 96, 192 i 320 kb/s. Liczymy, o ile kodek przesuwa uśredniony wektor cech, i
porównujemy to z różnicą między dwoma mówcami czytającymi to samo zdanie (`Speaker_6_153.wav`). Wartość 1,0 oznacza,
że kodek zmienia cechy tak samo mocno jak zmiana mówcy.

> **Przewidywanie 7.1** *(zapisz przed uruchomieniem kolejnej komórki; za błędną hipotezę nie traci się punktów)*: Od jakiej przepływności zmiana cech będzie mała w porównaniu ze zmianą mówcy? Czy przy 8 kb/s kodek zmieni cechy bardziej niż zmiana mówcy?
>
> *Moja hipoteza:*

In [ ]:
for nazwa in ['Speaker_8_153.flac', 'Speaker_8_153_8.mp3']:
    print(nazwa); display(Audio(wczytaj(DANE / nazwa), rate=SR))

srednia = lambda nazwa, k: extract(wczytaj(DANE / nazwa), k).mean(axis=0)     # uśredniony wektor cech nagrania
kodek = {}
for k in KINDS:
    oryginal = srednia('Speaker_8_153.flac', k)
    zmiana_mowcy = np.linalg.norm(oryginal - srednia('Speaker_6_153.wav', k))
    kodek[k] = {f'MP3 {kb} kb/s': np.linalg.norm(srednia(f'Speaker_8_153_{kb}.mp3', k) - oryginal) / zmiana_mowcy
                for kb in [8, 96, 192, 320]}
print("Przesunięcie cech przez kodek, jako ułamek różnicy między dwoma mówcami:")
display(pd.DataFrame(kodek).T.round(2))

### Krok 2: Czy model rozpoznaje głos, czy poziom nagrania?
Bierzemy model rozpoznawania osoby z Zadania 5 (uczenie PL, test EN) i zmieniamy tylko głośność nagrań testowych:
nagrania A wzmacniamy tak, żeby miały poziom mowy B, a nagrania B tak, żeby miały poziom A (według mediany z
metadanych). Głos się nie zmienia, zmienia się tylko poziom. Dla porównania uczymy też model, który widzi wyłącznie
$c_0$, czyli poziom nagrania.

> **Przewidywanie 7.2** *(zapisz przed uruchomieniem kolejnej komórki; za błędną hipotezę nie traci się punktów)*: Jak zamiana poziomów zmieni wynik modelu, który widzi tylko $c_0$, a jak modeli MFCC z $c_0$ i bez $c_0$?
>
> *Moja hipoteza:*

In [ ]:
poziom = rec.groupby('osoba')['poziom_mowy_dbfs'].median()
roznica = poziom['A'] - poziom['B']
print(f"Mediana poziomu mowy: A {poziom['A']:.1f} dBFS, B {poziom['B']:.1f} dBFS, różnica {roznica:.1f} dB")
wzmocnienie = {'A': -roznica, 'B': roznica}                   # dB: A dostaje poziom B, B dostaje poziom A
F_zam = {i: extract(signals[i] * 10 ** (wzmocnienie[rec.osoba[i]] / 20), 'MFCC') for i in np.flatnonzero(te)}
X_zam = X['MFCC'].copy()
for i, F in F_zam.items():
    X_zam[i] = pool_meanstd(F)

c0 = {'MFCC (z c0)': slice(None),
      'MFCC bez c0': np.r_[1:N_MFCC, N_MFCC + 1:2 * N_MFCC],      # bez średniej i odchylenia c0
      'tylko c0 (poziom)': [0, N_MFCC]}                          # wyłącznie średnia i odchylenie c0
poziomy = {}
for nazwa, cols in c0.items():
    m = make_clf().fit(X['MFCC'][tr][:, cols], y_osoba[tr])
    poziomy[nazwa] = {'test bez zmian': balanced_accuracy_score(y_osoba[te], m.predict(X['MFCC'][te][:, cols])),
                      'test z zamienionymi poziomami': balanced_accuracy_score(y_osoba[te], m.predict(X_zam[te][:, cols]))}
display(pd.DataFrame(poziomy).T.round(3))

### Krok 3: Szum, pasmo telefoniczne i augmentacja
Uczymy rozpoznawanie osoby (uczenie PL, test EN) na czystych nagraniach, a testujemy na zmienionych kopiach nagrań
testowych: z szumem (SNR 10 dB) i obciętych do pasma telefonicznego 300–3400 Hz. Potem sprawdzamy augmentację:
model uczymy na nagraniach czystych **i** zmienionych, ale tylko ze zbioru uczącego. **TODO 7.** Uzupełnij jedną linię w pętli.
Obliczenia trwają kilka minut. Nagłówki kolumn w wynikach czytaj tak: na czym model się uczył -> na czym
go testujemy.

> **Przewidywanie 7.3** *(zapisz przed uruchomieniem kolejnej komórki; za błędną hipotezę nie traci się punktów)*: Która reprezentacja najbardziej straci na szumie, a która na paśmie telefonicznym? Ile odzyska augmentacja?
>
> *Moja hipoteza:*

In [ ]:
TEL = scipy.signal.butter(6, [300, 3400], btype='bandpass', fs=SR, output='sos')
ZAKLOCENIA = {'szum, SNR 10 dB': lambda y: add_noise(y, 10), 'pasmo telefoniczne': lambda y: scipy.signal.sosfilt(TEL, y)}

rows = []
for zname, zmiana in ZAKLOCENIA.items():
    FZ = {k: [extract(zmiana(s).astype(np.float32), k) for s in signals] for k in KINDS}    # cechy zmienionych nagrań
    for rname, (k, cols) in REPS.items():
        Xc = np.array([pool_meanstd(F[:, cols]) for F in FR[k]])       # czyste
        Xz = np.array([pool_meanstd(F[:, cols]) for F in FZ[k]])       # zmienione
        m = make_clf().fit(Xc[tr], y_osoba[tr])
        # ===== TODO 7 (1 linia): wpisz kod w miejsce linii raise =====
        # X_aug = czyste Xc[tr] i zmienione Xz[tr] razem (np.vstack); y_aug = etykiety y_osoba[tr] dwa razy
        raise NotImplementedError("TODO 7: uzupełnij kod")
        # ===== koniec TODO 7 =====
        m_aug = make_clf().fit(X_aug, y_aug)
        rows.append({'zakłócenie': zname, 'reprezentacja': rname,
                     'czyste -> czyste': balanced_accuracy_score(y_osoba[te], m.predict(Xc[te])),
                     'czyste -> zmienione': balanced_accuracy_score(y_osoba[te], m.predict(Xz[te])),
                     'augm. -> zmienione': balanced_accuracy_score(y_osoba[te], m_aug.predict(Xz[te]))})
    print('gotowe:', zname)
shift_df = pd.DataFrame(rows).set_index(['zakłócenie', 'reprezentacja'])
display(shift_df.round(3))

fig, ax = plt.subplots(1, 3, figsize=(15, 2.8), sharey=True)
for a, col in zip(ax, shift_df.columns):
    sns.heatmap(shift_df[col].unstack(1).loc[list(ZAKLOCENIA), list(REPS)], annot=True, fmt='.2f', vmin=0.4, vmax=1,
                cmap='RdYlGn', ax=a, cbar=False)
    a.set(title=col, xlabel='', ylabel='')
plt.tight_layout(); plt.show()

### Pytania do Zadania 7

**P7.1** Krok 1: przy jakiej przepływności kodek zmienia cechy bardziej niż zmiana mówcy? Co to oznacza dla modelu rozpoznawania osoby uczonego na nagraniach studyjnych, a używanego na nagraniach z telefonu lub komunikatora?

**P7.2** Krok 2: jak zamiana poziomów zmieniła wynik każdego z trzech modeli? Wyjaśnij to: prześledź, co mnożenie sygnału przez stałą robi z cechami log-mel, a co potem robi z tym DCT. Czy pełny model MFCC korzystał z poziomu nagrania, czy z głosu? Porównaj różnicę median poziomów (wypisaną nad tabelą) z wynikiem modelu „tylko c0”: jeśli różnica jest mała, a ten model i tak odróżnia osoby, co poza poziomem mowy może nieść $c_0$? Powiąż to z klasyfikatorem na metadanych z Zadania 1 (poziom mowy, szum, SNR). Kiedy taki skrót byłby groźny?

**P7.3** Krok 3: która reprezentacja najbardziej straciła na każdym zakłóceniu i dlaczego? Ile odzyskała augmentacja? Przed jakimi zmianami warunków augmentacja nie ochroni modelu?

> **Twoja odpowiedź:**
>
> P7.1:
>
>
> P7.2:
>
>
> P7.3:

---
## Podsumowanie (1 pkt)

Uzupełnij tabelę Waszymi wynikami (konkretna liczba lub obserwacja) i regułą, którą zastosujesz w dowolnym projekcie
uczenia maszynowego.

| Zadanie | Co zaobserwowaliśmy (liczba lub obserwacja) | Reguła praktyczna |
|---|---|---|
| 1. Metadane nagrań | *[uzupełnij]* | *[uzupełnij]* |
| 2. To samo zdanie w trzech reprezentacjach | *[uzupełnij]* | *[uzupełnij]* |
| 3. Losowy podział fragmentów a test na drugiej osobie | *[uzupełnij]* | *[uzupełnij]* |
| 4. Rozpoznawanie zdań drugiej osoby (DTW) | *[uzupełnij]* | *[uzupełnij]* |
| 5. Rozpoznawanie osoby w innym języku | *[uzupełnij]* | *[uzupełnij]* |
| 6. Normalizacja | *[uzupełnij]* | *[uzupełnij]* |
| 7. Kodek, poziom, szum, augmentacja | *[uzupełnij]* | *[uzupełnij]* |

---
**Literatura**
- S. Davis, P. Mermelstein (1980), *Comparison of parametric representations for monosyllabic word recognition in
  continuously spoken sentences*, IEEE Trans. ASSP 28(4)
- H. Sakoe, S. Chiba (1978), *Dynamic programming algorithm optimization for spoken word recognition*, IEEE Trans.
  ASSP 26(1) (DTW)
- S. Kapoor, A. Narayanan (2023), *Leakage and the reproducibility crisis in machine-learning-based science*,
  Patterns 4(9) (o wycieku danych przy walidacji)
- R. Geirhos i in. (2020), *Shortcut learning in deep neural networks*, Nature Machine Intelligence 2
- D. Park i in. (2019), *SpecAugment: A simple data augmentation method for automatic speech recognition*, Interspeech
- Dokumentacja: [librosa](https://librosa.org/doc/latest/),
  [scikit-learn: walidacja z grupami](https://scikit-learn.org/stable/modules/cross_validation.html#group-k-fold)